### Structured output
*Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.*

#### Pydantic
*Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.*

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023D7F7DC280>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023D7F7DCAF0>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field("The title of the movie")
    year:int = Field("The Release date of movie")
    director:str= Field("The Director of Movie")
    rating:float = Field("The rating of the movie")


In [14]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023D7F7DC280>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023D7F7DCAF0>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'defa

In [4]:
model.invoke("inception")

AIMessage(content='**Inception** (2010) – Directed by Christopher\u202fNolan\n\n| Element | Details |\n|---------|---------|\n| **Genre** | Sci‑fi action‑thriller |\n| **Main Cast** | Leonardo\u202fDiCaprio (Dom\u202fCobb), Joseph\u202fGordon‑Levitt (Arthur), Ellen\u202fPage (Ariadne), Tom\u202fHardy (Eames), Ken\u202fWatanabe (Yusuf), Dileep\u202fNair (Saito), Michael\u202fCaine (Robert\u202fSaito), Cillian\u202fMurphy (Malcolm), Tom\u202fHardy (Eames), etc. |\n| **Premise** | Dom\u202fCobb is a “extractor” who enters people’s dreams to steal secrets. He’s offered a chance to erase his criminal record by performing the reverse: planting an idea in someone’s mind (inception). |\n| **Plot Highlights** | 1. **Setup** – Cobb’s team is hired by Saito to plant an idea in the mind of Robert Fischer, the heir to a business empire, to dissolve his father’s company. 2. **Dream Layers** – The team constructs three nested dream levels: a hotel (first level), a train (second), and a snowy fortress

In [15]:
responce = model_with_structured_output.invoke("inception")
responce

Movie(title='Inception', year=2010, director='Christopher Nolan')

#### Message output alongside parsed structure 

In [19]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ Movie details """
    title:str= Field("The title of the movie")
    year:int = Field("The Release date of movie")
    director:str= Field("The Director of Movie")
    rating:float = Field("The rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw= True)
model_with_structure.invoke("DDLJ")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User says "DDLJ". Likely they want details about the movie "Dilwale Dulhania Le Jayenge". They might want movie details. We can call the function. Provide director, rating, title, year. Use defaults? We can provide actual data. Let\'s call function with title "Dilwale Dulhania Le Jayenge". Probably director: "Aditya Chopra". Year: 1995. Rating: maybe 8.5? Let\'s provide.', 'tool_calls': [{'id': 'fc_fab6f61f-ae91-4f27-be90-c68a5fa313b9', 'function': {'arguments': '{"director":"Aditya Chopra","rating":8.5,"title":"Dilwale Dulhania Le Jayenge","year":1995}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 168, 'total_tokens': 309, 'completion_time': 0.14621308, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.008189599, 'prompt_tokens_details': None, 'queue_time': 0.332107659, 'total_time': 0.154402679}, 'model_name': 'openai/

#### Nested Structure

In [22]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title : str
    genre : list[str]
    cast : list[Actor]
    director : str
    rating : float
    budget : float | None = Field("Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)
model_with_structure.invoke("DDLJ")

MovieDetails(title='Dilwale Dulhania Le Jayenge', genre=['Romance', 'Drama'], cast=[Actor(name='Shah Rukh Khan', role='Raj Malhotra'), Actor(name='Kajol', role='Simran')], director='Aditya Chopra', rating=8.5, budget=None)

#### TypedDict
*TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.*

In [23]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
model_with_structure.invoke("Avengers")

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [25]:
class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title : str
    genre : list[str]
    cast : list[Actor]
    director : str
    rating : float
    budget : float 

model_with_structure = model.with_structured_output(MovieDetails)
model_with_structure.invoke("Avengers")

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'director': 'Joss Whedon',
 'genre': ['Action', 'Adventure', 'Sci-Fi'],
 'rating': 8,
 'title': 'Avengers'}

In [26]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

#### DataClasses
*A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator*

In [31]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [32]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

result = model.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.